# Memory-Efficient Multi-Domain LoRA Routing

This notebook trains three small LoRA adapters—`code`, `medical`, and
`general`—against one shared GPT-2 base model. A TF-IDF + Logistic
Regression router predicts the domain of each prompt, and the inference
wrapper activates the matching adapter without loading three full models.

In [ ]:
%pip uninstall -y torchao
%pip install -q \
    "transformers==4.48.3" \
    "peft==0.14.0" \
    "datasets==3.2.0" \
    "accelerate==1.3.0" \
    "scikit-learn==1.6.1" \
    "joblib==1.4.2" \
    "safetensors>=0.4.5"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency c

In [10]:
import gc
import json
import os
import random
from pathlib import Path

import datasets
import joblib
import numpy as np
import peft
import sklearn
import torch
import transformers
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
MODEL_ID = "gpt2"
DOMAINS = ("code", "medical", "general")
SAMPLES_PER_DOMAIN = 1_000
MAX_LENGTH = 384
MIN_RESPONSE_TOKENS = 96
FORCE_RETRAIN = False

PROJECT_DIR = Path("/content/multidomain_lora")
ADAPTER_ROOT = PROJECT_DIR / "adapters"
CHECKPOINT_ROOT = PROJECT_DIR / "checkpoints"
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
ROUTER_PATH = PROJECT_DIR / "router.joblib"

for directory in (PROJECT_DIR, ADAPTER_ROOT, CHECKPOINT_ROOT, TOKENIZER_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32



## Load and normalize the datasets

Source datasets:

- Code: [`sahil2801/CodeAlpaca-20k`](https://huggingface.co/datasets/sahil2801/CodeAlpaca-20k)
- Medical: [`lavita/MedQuAD`](https://huggingface.co/datasets/lavita/MedQuAD)
- General: [`tatsu-lab/alpaca`](https://huggingface.co/datasets/tatsu-lab/alpaca)



In [ ]:
DATASET_IDS = {
    "code": "sahil2801/CodeAlpaca-20k",
    "medical": "lavita/MedQuAD",
    "general": "tatsu-lab/alpaca",
}


def clean_text(value):
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"none", "nan", "null"} else text


def add_optional_input(instruction, extra_input):
    instruction = clean_text(instruction)
    extra_input = clean_text(extra_input)
    if extra_input and extra_input.lower() not in {"<noinput>", "<no input>"}:
        return f"{instruction}\n\nInput:\n{extra_input}"
    return instruction


def normalize_example(domain, example):
    if domain in {"code", "general"}:
        required = {"instruction", "output"}
        missing = required.difference(example)
        if missing:
            raise KeyError(
                f"{domain} dataset is missing columns {sorted(missing)}; "
                f"available columns: {sorted(example)}"
            )
        prompt = add_optional_input(example["instruction"], example.get("input", ""))
        response = clean_text(example["output"])
    elif domain == "medical":
        required = {"question", "answer"}
        missing = required.difference(example)
        if missing:
            raise KeyError(
                f"medical dataset is missing columns {sorted(missing)}; "
                f"available columns: {sorted(example)}"
            )
        prompt = clean_text(example["question"])
        response = clean_text(example["answer"])
    else:
        raise ValueError(f"Unknown domain: {domain}")

    if not prompt or not response:
        return None

    text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"
    return {
        "domain": domain,
        "prompt": prompt,
        "response": response,
        "text": text,
    }


def load_domain_dataset(domain, sample_count=SAMPLES_PER_DOMAIN):
    source = load_dataset(DATASET_IDS[domain], split="train")
    source = source.shuffle(seed=SEED)

    records = []
    for row in source:
        normalized = normalize_example(domain, row)
        if normalized is not None:
            records.append(normalized)
        if len(records) >= sample_count:
            break

    if len(records) < sample_count:
        raise RuntimeError(
            f"Only {len(records)} valid {domain} examples were found; "
            f"requested {sample_count}."
        )
    return Dataset.from_list(records)


domain_datasets = {
    domain: load_domain_dataset(domain)
    for domain in DOMAINS
}

for domain, dataset in domain_datasets.items():
    print(f"{domain:>7}: {len(dataset):,} examples | columns={dataset.column_names}")
    print(" sample:", dataset[0]["prompt"][:140].replace("\n", " "), "\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

   code: 1,000 examples | columns=['domain', 'prompt', 'response', 'text']
 sample: Design a class for representing a person in Python. 

medical: 1,000 examples | columns=['domain', 'prompt', 'response', 'text']
 sample: How to prevent Marburg hemorrhagic fever (Marburg HF) ? 

general: 1,000 examples | columns=['domain', 'prompt', 'response', 'text']
 sample: What would be the best type of exercise for a person who has arthritis? 



In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.save_pretrained(TOKENIZER_DIR)


def prompt_prefix(prompt):
    return f"### Instruction:\n{prompt}\n\n### Response:\n"


def tokenize_supervised(example):
    prompt_ids = tokenizer(
        prompt_prefix(example["prompt"]),
        add_special_tokens=False,
    )["input_ids"]
    response_ids = tokenizer(
        example["response"] + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    response_reserve = min(MIN_RESPONSE_TOKENS, len(response_ids))
    max_prompt_tokens = max(1, MAX_LENGTH - response_reserve)
    prompt_ids = prompt_ids[:max_prompt_tokens]
    response_ids = response_ids[: MAX_LENGTH - len(prompt_ids)]

    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + response_ids.copy()

    padding = MAX_LENGTH - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding
    attention_mask += [0] * padding
    labels += [-100] * padding

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


tokenized_datasets = {
    domain: dataset.map(
        tokenize_supervised,
        remove_columns=dataset.column_names,
        desc=f"Tokenizing {domain}",
    )
    for domain, dataset in domain_datasets.items()
}

for domain, dataset in tokenized_datasets.items():
    supervised_tokens = sum(token != -100 for token in dataset[0]["labels"])
    print(f"{domain:>7}: {len(dataset):,} rows; first row has {supervised_tokens} response tokens")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing code:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing medical:   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1553 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing general:   0%|          | 0/1000 [00:00<?, ? examples/s]

   code: 1,000 rows; first row has 139 response tokens
medical: 1,000 rows; first row has 360 response tokens
general: 1,000 rows; first row has 40 response tokens


In [ ]:
REQUIRED_ADAPTER_FILES = (
    "adapter_config.json",
    "adapter_model.safetensors",
)


def validate_adapter_dir(adapter_dir, expected_base=MODEL_ID):
    adapter_dir = Path(adapter_dir)
    missing = [
        name for name in REQUIRED_ADAPTER_FILES
        if not (adapter_dir / name).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Invalid PEFT adapter directory {adapter_dir}: missing {missing}. "
            "Pass the exact directory created by model.save_pretrained()."
        )

    with (adapter_dir / "adapter_config.json").open(encoding="utf-8") as handle:
        config = json.load(handle)
    saved_base = config.get("base_model_name_or_path")
    if saved_base and saved_base != expected_base:
        raise ValueError(
            f"Adapter {adapter_dir} expects base model {saved_base!r}, "
            f"but this project uses {expected_base!r}."
        )
    return True


def adapter_is_valid(adapter_dir):
    try:
        return validate_adapter_dir(adapter_dir)
    except (FileNotFoundError, ValueError, json.JSONDecodeError):
        return False


def train_one_adapter(domain, train_dataset):
    if domain not in DOMAINS:
        raise ValueError(f"Unknown domain {domain!r}; expected one of {DOMAINS}")

    adapter_dir = ADAPTER_ROOT / domain
    if adapter_is_valid(adapter_dir) and not FORCE_RETRAIN:
        print(f"Skipping {domain}: valid adapter already exists at {adapter_dir}")
        return {"status": "skipped", "adapter_dir": str(adapter_dir)}

    print(f"\nTraining {domain!r} adapter...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=TRAIN_DTYPE,
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.use_cache = False
    if hasattr(base_model, "gradient_checkpointing_disable"):
        base_model.gradient_checkpointing_disable()

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["c_attn", "c_proj"],
        fan_in_fan_out=True,
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=str(CHECKPOINT_ROOT / domain),
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        weight_decay=0.01,
        logging_steps=25,
        save_strategy="no",
        report_to="none",
        fp16=(DEVICE == "cuda"),
        bf16=False,
        optim="adamw_torch",
        gradient_checkpointing=False,
        dataloader_pin_memory=(DEVICE == "cuda"),
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=default_data_collator,
    )
    train_result = trainer.train()

    model.save_pretrained(adapter_dir, safe_serialization=True)
    validate_adapter_dir(adapter_dir)
    print(f"Saved and validated: {adapter_dir}")

    metrics = dict(train_result.metrics)
    metrics.update({"status": "trained", "adapter_dir": str(adapter_dir)})

    del trainer, model, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics

In [ ]:
training_results = {}
for domain in DOMAINS:
    training_results[domain] = train_one_adapter(
        domain,
        tokenized_datasets[domain],
    )

print("\nAdapter summary")
for domain, result in training_results.items():
    print(f"{domain:>7}: {result['status']} -> {result['adapter_dir']}")


Training 'code' adapter...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


...

Saved and validated: /content/multidomain_lora/adapters/code

Training 'medical' adapter...
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


Step,Training Loss
25,2.728000
50,2.626200


Saved and validated: /content/multidomain_lora/adapters/medical

Training 'general' adapter...
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


Step,Training Loss
25,2.770500
50,2.538600


Saved and validated: /content/multidomain_lora/adapters/general

Adapter summary
   code: trained -> /content/multidomain_lora/adapters/code
medical: trained -> /content/multidomain_lora/adapters/medical
general: trained -> /content/multidomain_lora/adapters/general


In [ ]:
router_prompts = []
router_labels = []
for domain, dataset in domain_datasets.items():
    router_prompts.extend(dataset["prompt"])
    router_labels.extend([domain] * len(dataset))

router_training_path = PROJECT_DIR / "router_training.jsonl"
Dataset.from_dict({
    "prompt": router_prompts,
    "domain": router_labels,
}).to_json(router_training_path)
print(f"Saved router training records: {router_training_path}")

X_train, X_test, y_train, y_test = train_test_split(
    router_prompts,
    router_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=router_labels,
)


def build_router():
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=40_000,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1_000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


evaluation_router = build_router()
evaluation_router.fit(X_train, y_train)
predictions = evaluation_router.predict(X_test)

print(f"Held-out accuracy: {accuracy_score(y_test, predictions):.3f}")
print(classification_report(y_test, predictions, digits=3))
print("Confusion matrix; rows=true, columns=predicted")
print("labels:", DOMAINS)
print(confusion_matrix(y_test, predictions, labels=DOMAINS))

router = build_router()
router.fit(router_prompts, router_labels)
router_artifact = {
    "pipeline": router,
    "domains": DOMAINS,
    "base_model": MODEL_ID,
    "seed": SEED,
    "training_samples": len(router_prompts),
}
joblib.dump(router_artifact, ROUTER_PATH)
print(f"Saved router: {ROUTER_PATH}")

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Saved router training records: /content/multidomain_lora/router_training.jsonl
Held-out accuracy: 0.923
              precision    recall  f1-score   support

        code      0.886     0.890     0.888       200
     general      0.889     0.880     0.884       200
     medical      0.995     1.000     0.998       200

    accuracy                          0.923       600
   macro avg      0.923     0.923     0.923       600
weighted avg      0.923     0.923     0.923       600

Confusion matrix; rows=true, columns=predicted
labels: ('code', 'medical', 'general')
[[178   0  22]
 [  0 200   0]
 [ 23   1 176]]
Saved router: /content/multidomain_lora/router.joblib


In [ ]:
class DynamicLoRARouter:
    def __init__(self, project_dir=PROJECT_DIR, device=None):
        self.project_dir = Path(project_dir)
        self.adapter_root = self.project_dir / "adapters"
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

        router_artifact = joblib.load(self.project_dir / "router.joblib")
        self.router = router_artifact["pipeline"]
        self.domains = tuple(router_artifact["domains"])
        self.base_model_id = router_artifact["base_model"]

        if set(self.domains) != set(DOMAINS):
            raise ValueError(
                f"Router domains {self.domains} do not match expected domains {DOMAINS}."
            )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.project_dir / "tokenizer",
            use_fast=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        adapter_paths = {
            domain: self.adapter_root / domain
            for domain in self.domains
        }
        for path in adapter_paths.values():
            validate_adapter_dir(path, expected_base=self.base_model_id)

        base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_id,
            torch_dtype=self.dtype,
        )
        base_model.config.pad_token_id = self.tokenizer.pad_token_id
        base_model.config.use_cache = True
        base_model.to(self.device)

        first_domain = self.domains[0]
        self.model = PeftModel.from_pretrained(
            base_model,
            adapter_paths[first_domain],
            adapter_name=first_domain,
            is_trainable=False,
        )
        for domain in self.domains[1:]:
            self.model.load_adapter(
                adapter_paths[domain],
                adapter_name=domain,
                is_trainable=False,
            )

        self.model.eval()
        self.model.to(self.device)
        print(f"Loaded one {self.base_model_id!r} base model with adapters: {self.domains}")

    def route(self, prompt):
        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError("prompt must be a non-empty string")
        probabilities = self.router.predict_proba([prompt])[0]
        classes = self.router.named_steps["classifier"].classes_
        best_index = int(np.argmax(probabilities))
        return {
            "domain": str(classes[best_index]),
            "confidence": float(probabilities[best_index]),
            "probabilities": {
                str(label): float(probability)
                for label, probability in zip(classes, probabilities)
            },
        }

    def generate(
        self,
        prompt,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.7,
        top_p=0.9,
    ):
        routing = self.route(prompt)
        selected_domain = routing["domain"]
        self.model.set_adapter(selected_domain)
        self.model.eval()

        formatted_prompt = prompt_prefix(prompt)
        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(self.device)

        generation_args = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
        }
        if do_sample:
            generation_args.update({"temperature": temperature, "top_p": top_p})

        with torch.inference_mode():
            output_ids = self.model.generate(**inputs, **generation_args)

        new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return {
            **routing,
            "response": response,
        }

In [ ]:
engine = DynamicLoRARouter(PROJECT_DIR)

test_prompts = [
    "Write a Python function that returns the unique values in a list while preserving order.",
    "What are common symptoms of iron-deficiency anemia?",
    "Write a polite two-sentence email asking to reschedule a meeting.",
]

for prompt in test_prompts:
    result = engine.generate(prompt, max_new_tokens=96, do_sample=False)
    print("=" * 80)
    print("PROMPT:", prompt)
    print(f"ROUTE: {result['domain']} ({result['confidence']:.1%})")
    print("ALL PROBABILITIES:", result["probabilities"])
    print("RESPONSE:", result["response"])

Loaded one 'gpt2' base model with adapters: ('code', 'medical', 'general')
PROMPT: Write a Python function that returns the unique values in a list while preserving order.
ROUTE: code (93.1%)
ALL PROBABILITIES: {'code': 0.9312308475373274, 'general': 0.05712018361146011, 'medical': 0.011648968851212258}
RESPONSE: Write a Python function that returns the unique values in a list while preserving order.
###
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
PROMPT: What are common symptoms of iron-deficiency anemia?
ROUTE: medical (95.5%)
ALL PROBABILITIES: {'code': 0.01632614067135088, 'general': 0.029100471313153433, 'medical': 0.9545733880154956}
RESPONSE: Iron deficiency anemia is a condition in which the body produces iron-containing proteins called iron-1 and iron-2. These proteins are found in the blood and are responsible for the production of iron. The body produces these proteins by taking in iron from the blood. The body also produces iron-containing 